# Exercise 8 : Grouping Data

Run the first cell : it reads `titanic.csv` into `titanic`, one row per passenger. The steps use
these columns :

- `pclass` : ticket class 1, 2 or 3
- `survived` : 1 or 0
- `sex`, `age`, `fare` and `cabin`
- `embarked` : the port where the passenger boarded, `S` Southampton, `C` Cherbourg or `Q`
  Queenstown

In [ ]:
from pathlib import Path

import pandas as pd

DATA = Path.cwd().parents[1] / 'datasets' / 'exercises'
titanic = pd.read_csv(DATA / 'titanic.csv')
titanic.head()

## Step 1 : Counting

Group `titanic` by `pclass` into `by_class` and count the non-missing `survived` values per class
as `count_by_class`. Group by `embarked` and `pclass` together and count `survived` again as
`count_mult`. Print both.

In [ ]:
by_class = titanic.groupby('pclass')
count_by_class = by_class['survived'].count()
print(count_by_class)

by_mult = titanic.groupby(['embarked', 'pclass'])
count_mult = by_mult['survived'].count()
print(count_mult)

## Step 2 : size and count

`count()` counts non-missing values ; `size()` counts rows. Print `by_class.size()` and
`by_class['age'].count()`. How many passengers have no recorded age?

`count_mult` adds up to 1307 of the 1309 passengers. Group by `['embarked', 'pclass']` again with
`dropna=False`, compute `size()` as `size_mult` and check that `size_mult.sum()` equals
`len(titanic)`. Which passengers were missing from `count_mult`?

In [ ]:
print(by_class.size())
print(by_class['age'].count())
print(len(titanic) - titanic['age'].count())

print(count_mult.sum())
size_mult = titanic.groupby(['embarked', 'pclass'], dropna=False).size()
print(size_mult)
print(size_mult.sum() == len(titanic))

## Step 3 : Empty categories

Convert `embarked` to a categorical with the categories `port_order`, the order in which the
Titanic called at the ports. Select the passengers aged 60 or over as `seniors`.

Group `seniors` by `['embarked', 'pclass']` and count `survived`, once with `observed=True` and
once with `observed=False`. Which combinations appear only with `observed=False`? How does their 0
differ from a measured zero?

In [ ]:
port_order = ['S', 'C', 'Q']

In [ ]:
ports = titanic.copy()
ports['embarked'] = pd.Categorical(ports['embarked'], categories=port_order)
seniors = ports.loc[ports['age'] >= 60]
print(seniors.groupby(['embarked', 'pclass'], observed=True)['survived'].count())
print(seniors.groupby(['embarked', 'pclass'], observed=False)['survived'].count())

## Step 4 : Several aggregates

Group by `pclass`, select `age` and `fare`, and aggregate with `['max', 'median']`. Print the oldest
age per class with `.loc[:, ('age', 'max')]` and the median fare per class with
`.loc[:, ('fare', 'median')]`.

Compute the same two results with named aggregation, `oldest_age=('age', 'max')` and
`median_fare=('fare', 'median')`, and print them.

In [ ]:
aggregated = by_class[['age', 'fare']].agg(['max', 'median'])
print(aggregated.loc[:, ('age', 'max')])
print(aggregated.loc[:, ('fare', 'median')])

named = by_class.agg(oldest_age=('age', 'max'), median_fare=('fare', 'median'))
print(named)

## Step 5 : Filling by group

Fill each missing age with the median age of the passengers of the same sex and class. Work on a
copy, `filled = titanic.copy()`, so that `titanic` keeps its missing ages for Step 7 :

- Save a copy of `filled['age']` as `original_age`
- Add a column `age_imputed` that is `True` where the age is missing
- Write a function `impute_median(series)` that returns `series.fillna(series.median())`
- Group `filled` by `['sex', 'pclass']` and assign the result of `transform(impute_median)` on
  `age` back to `filled['age']`

Print the number of filled ages. Check that every known age is unchanged : compare
`filled.loc[~filled['age_imputed'], 'age']` with `original_age.dropna()` using `.equals()`. Why
keep the `age_imputed` column for later analyses?

In [ ]:
filled = titanic.copy()
original_age = filled['age'].copy()
filled['age_imputed'] = original_age.isna()


def impute_median(series):
    return series.fillna(series.median())


filled['age'] = filled.groupby(['sex', 'pclass'])['age'].transform(impute_median)
print(filled['age_imputed'].sum())
print(filled.loc[~filled['age_imputed'], 'age'].equals(original_age.dropna()))

## Step 6 : Apply

The notebook defines `c_deck_survival(group)`, the survival rate of the passengers in a group whose
cabin is on deck C. Group by `sex`, select the `cabin` and `survived` columns and call
`.apply(c_deck_survival)`.

In [ ]:
def c_deck_survival(group):
    c_passengers = group['cabin'].str.startswith('C')
    return group.loc[c_passengers, 'survived'].mean()

In [ ]:
c_surv_by_sex = titanic.groupby('sex')[['cabin', 'survived']].apply(c_deck_survival)
print(c_surv_by_sex)

## Step 7 : Derived groups

Which share of the children under 10 survived in each class?

Create the Boolean Series `titanic['age'] < 10` and map it to labels with
`.map({True: 'under 10', False: '10 or older'})`. A missing age compares as `False` and would land
in `'10 or older'` : relabel those passengers with `.where(titanic['age'].notna(), 'age unknown')`.
Name the result `under10` and print its `value_counts()`.

Group by `under10` and aggregate `survived` with `['mean', 'size']`. Group by `under10` and
`pclass` together. Do the passenger counts add up to 1309?

In [ ]:
under10 = (titanic['age'] < 10).map({True: 'under 10', False: '10 or older'})
under10 = under10.where(titanic['age'].notna(), 'age unknown')
print(under10.value_counts())

survival_by_age = titanic.groupby(under10)['survived'].agg(['mean', 'size'])
print(survival_by_age)

survival_by_age_class = titanic.groupby([under10, 'pclass'])['survived'].agg(['mean', 'size'])
print(survival_by_age_class)
print(survival_by_age_class['size'].sum())

## Step 8 : Filtering groups

The first letter of `cabin` is the deck : `titanic['cabin'].str[0]`. Store it as a column `deck`.

Keep the decks with at least 20 passengers with `groupby('deck').filter(...)`. Compute the survival
rate and the number of passengers per remaining deck with `agg(['mean', 'size'])`, and sort the
decks from the highest to the lowest survival rate. How many passengers have no deck?

In [ ]:
decks = titanic.assign(deck=titanic['cabin'].str[0])
common = decks.groupby('deck').filter(lambda group: len(group) >= 20)
by_deck = common.groupby('deck')['survived'].agg(['mean', 'size'])
print(by_deck.sort_values('mean', ascending=False))
print(decks['deck'].isna().sum())